In [5]:
import os
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

In [20]:
import pandas as pd
import numpy as np

#df = pd.read_excel(r"C:\Users\Usuario\Downloads\Listado normas - palabra clave 1869-2023.xlsx")

df = pd.read_excel(r"C:\Users\Usuario\Downloads\Normativa nacional víctimas 1869-2023.xlsx")

#df['caso_ok'] = np.where(df['derechos'] == "no", 0, 1)

# Elimina la columna y guarda el resultado en el mismo DataFrame
#df = df.drop(['derechos', 'texto completo', 'Resumen'], axis=1)



df_nuevo = df[['Título','caso_ok']].copy()

df_nuevo

,Título,caso_ok
0,CODIGO CIVIL,0
1,SUBSIDIO ESTATAL,1
2,SUBSIDIO ESTATAL,1
3,CODIGO PROCESAL PENAL DE LA NACION,0
4,CODIGO DE COMERCIO,0
...,...,...
199,ACUERDOS,1
200,ACUERDOS,1
201,PROGRAMA MÉDICO OBLIGATORIO DE LAS OBRAS SOCIA...,0
202,CORTE INTERAMERICANA DE DERECHOS HUMANOS,1


In [21]:
from sklearn.model_selection import train_test_split

# 1. Definir X (todas las columnas menos la que querés predecir) e y (la columna objetivo)
# REEMPLAZA 'Nombre_De_Tu_Columna' por el nombre real en tu Excel
X = df_nuevo.drop(['Título'], axis=1) 
y = df_nuevo['caso_ok']

# 2. Dividir: el 80% para entrenar (train) y el 20% para probar (test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# 3. Verificar los tamaños
print(f"Registros para entrenamiento: {len(X_train)}")
print(f"Registros para prueba: {len(X_test)}")

Registros para entrenamiento: 163
Registros para prueba: 41


In [26]:
import logging
from transformers import logging as hf_logging

# Le decimos a Hugging Face que solo muestre errores, no reportes ni warnings
hf_logging.set_verbosity_error()

from transformers import pipeline
clasificador = pipeline("sentiment-analysis", model="pysentimiento/robertuito-sentiment-analysis")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [27]:
titulos = df_nuevo['Título'].astype(str).tolist()
resultados = clasificador(titulos)

In [28]:
df_nuevo['sentimiento'] = [res['label'] for res in resultados]

df_nuevo['confianza'] = [res['score'] for res in resultados]

df_nuevo

,Título,caso_ok,sentimiento,confianza
0,CODIGO CIVIL,0,NEU,0.757552
1,SUBSIDIO ESTATAL,1,NEU,0.577381
2,SUBSIDIO ESTATAL,1,NEU,0.577381
3,CODIGO PROCESAL PENAL DE LA NACION,0,NEU,0.775927
4,CODIGO DE COMERCIO,0,NEU,0.848917
...,...,...,...,...
199,ACUERDOS,1,NEU,0.516949
200,ACUERDOS,1,NEU,0.516949
201,PROGRAMA MÉDICO OBLIGATORIO DE LAS OBRAS SOCIA...,0,NEU,0.762960
202,CORTE INTERAMERICANA DE DERECHOS HUMANOS,1,NEU,0.725331


In [29]:
print(df_nuevo['sentimiento'].value_counts())
print(df_nuevo['confianza'].value_counts())

sentimiento
NEU    183
NEG     18
POS      3
Name: count, dtype: int64
confianza
0.577381    12
0.516949    10
0.628720     8
0.589180     7
0.725331     7
            ..
0.562435     1
0.672902     1
0.482219     1
0.685763     1
0.735164     1
Name: count, Length: 113, dtype: int64
